# Part C — Modeling and Hyperparameter Tuning

**Project:** Academy Nova Course Cancellation Prediction  
**Group:** 51  
**Main metric:** ROC-AUC  

This notebook covers model training, validation, hyperparameter tuning, and model comparison.

The main goal is to build a model with strong AUC while controlling overfitting.  
We will compare several models using cross-validation and track both validation performance and train-validation gaps.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.config import ID_COL, TARGET_COL, RANDOM_STATE, N_SPLITS
from src.data_loading import load_raw_data, validate_raw_data
from src.features import create_engineered_features

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

In [2]:
train_df, test_df = load_raw_data()
validate_raw_data(train_df, test_df)

train_fe = create_engineered_features(train_df)
test_fe = create_engineered_features(test_df)

print("Train with features:", train_fe.shape)
print("Test with features:", test_fe.shape)

Validating raw data...
Raw data validation passed.
Train shape: (63464, 29)
Test shape: (15866, 28)
Target positive rate: 0.4144
Train with features: (63464, 67)
Test with features: (15866, 66)


In [3]:
X = train_fe.drop(columns=[TARGET_COL])
y = train_fe[TARGET_COL]

X_test = test_fe.copy()

print("X shape:", X.shape)
print("y shape:", y.shape)
print("X_test shape:", X_test.shape)

X shape: (63464, 66)
y shape: (63464,)
X_test shape: (15866, 66)


## 1. Preprocessing Strategy

The dataset contains numeric, categorical, date, and ID-like columns.

For the baseline models:
- numeric features are imputed with the median,
- categorical features are imputed with `"Missing"` and one-hot encoded,
- unknown categories in validation/test are ignored,
- `Client_ID` is removed because it is only an identifier,
- `Course_Start_Date` is removed after extracting date-derived features.

This preprocessing is fitted only on the training folds inside cross-validation to avoid data leakage.

In [4]:
DROP_COLS = [ID_COL, "Course_Start_Date"]

X_model = X.drop(columns=[col for col in DROP_COLS if col in X.columns])
X_test_model = X_test.drop(columns=[col for col in DROP_COLS if col in X_test.columns])

numeric_cols = X_model.select_dtypes(include=["int64", "float64", "Int64"]).columns.tolist()
categorical_cols = X_model.select_dtypes(include=["object", "category"]).columns.tolist()

print("Numeric columns:", len(numeric_cols))
print("Categorical columns:", len(categorical_cols))
print("Total model columns:", X_model.shape[1])

Numeric columns: 50
Categorical columns: 10
Total model columns: 64


In [5]:
numeric_preprocessor = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_preprocessor = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value="Missing")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_preprocessor, numeric_cols),
        ("cat", categorical_preprocessor, categorical_cols),
    ],
    remainder="drop"
)

In [6]:
def evaluate_model_cv(model, X, y, model_name, n_splits=N_SPLITS):
    """
    Evaluate a model using Stratified K-Fold CV.

    Returns:
    - fold results table
    - out-of-fold predictions
    - fitted fold models
    """
    skf = StratifiedKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=RANDOM_STATE
    )

    oof_preds = np.zeros(len(X))
    fold_results = []
    fold_models = []

    for fold, (train_idx, valid_idx) in enumerate(skf.split(X, y), start=1):
        X_train, X_valid = X.iloc[train_idx], X.iloc[valid_idx]
        y_train, y_valid = y.iloc[train_idx], y.iloc[valid_idx]

        pipeline = Pipeline(steps=[
            ("preprocessor", preprocessor),
            ("model", model)
        ])

        pipeline.fit(X_train, y_train)

        train_pred = pipeline.predict_proba(X_train)[:, 1]
        valid_pred = pipeline.predict_proba(X_valid)[:, 1]

        train_auc = roc_auc_score(y_train, train_pred)
        valid_auc = roc_auc_score(y_valid, valid_pred)

        oof_preds[valid_idx] = valid_pred
        fold_models.append(pipeline)

        fold_results.append({
            "model": model_name,
            "fold": fold,
            "train_auc": train_auc,
            "valid_auc": valid_auc,
            "gap": train_auc - valid_auc
        })

        print(
            f"{model_name} | Fold {fold}: "
            f"train AUC={train_auc:.5f}, "
            f"valid AUC={valid_auc:.5f}, "
            f"gap={train_auc - valid_auc:.5f}"
        )

    fold_results_df = pd.DataFrame(fold_results)
    oof_auc = roc_auc_score(y, oof_preds)

    print("\nOOF AUC:", round(oof_auc, 5))
    print("Mean valid AUC:", round(fold_results_df["valid_auc"].mean(), 5))
    print("Std valid AUC:", round(fold_results_df["valid_auc"].std(), 5))
    print("Mean gap:", round(fold_results_df["gap"].mean(), 5))

    return fold_results_df, oof_preds, fold_models

## 2. Baseline Model Comparison

We first train several baseline models.  
The purpose is not only to get the highest AUC, but also to understand which model family fits this dataset best and how much overfitting each model shows.

In [7]:
log_reg = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=RANDOM_STATE
)

logreg_results, logreg_oof, logreg_models = evaluate_model_cv(
    log_reg,
    X_model,
    y,
    "Logistic Regression"
)

Logistic Regression | Fold 1: train AUC=0.91254, valid AUC=0.90685, gap=0.00569
Logistic Regression | Fold 2: train AUC=0.91265, valid AUC=0.90648, gap=0.00617
Logistic Regression | Fold 3: train AUC=0.91377, valid AUC=0.90165, gap=0.01211
Logistic Regression | Fold 4: train AUC=0.91404, valid AUC=0.90079, gap=0.01325
Logistic Regression | Fold 5: train AUC=0.91371, valid AUC=0.90259, gap=0.01112

OOF AUC: 0.90365
Mean valid AUC: 0.90367
Std valid AUC: 0.00281
Mean gap: 0.00967


In [8]:
rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=12,
    min_samples_leaf=20,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    class_weight="balanced"
)

rf_results, rf_oof, rf_models = evaluate_model_cv(
    rf,
    X_model,
    y,
    "Random Forest"
)

Random Forest | Fold 1: train AUC=0.88776, valid AUC=0.88974, gap=-0.00198
Random Forest | Fold 2: train AUC=0.89006, valid AUC=0.88890, gap=0.00116
Random Forest | Fold 3: train AUC=0.89354, valid AUC=0.88479, gap=0.00874
Random Forest | Fold 4: train AUC=0.89257, valid AUC=0.88282, gap=0.00976
Random Forest | Fold 5: train AUC=0.88511, valid AUC=0.88622, gap=-0.00111

OOF AUC: 0.88628
Mean valid AUC: 0.88649
Std valid AUC: 0.00286
Mean gap: 0.00331


In [9]:
hgb = HistGradientBoostingClassifier(
    learning_rate=0.05,
    max_iter=300,
    max_leaf_nodes=31,
    l2_regularization=0.1,
    random_state=RANDOM_STATE
)

hgb_results, hgb_oof, hgb_models = evaluate_model_cv(
    hgb,
    X_model,
    y,
    "HistGradientBoosting"
)

HistGradientBoosting | Fold 1: train AUC=0.96137, valid AUC=0.94916, gap=0.01221
HistGradientBoosting | Fold 2: train AUC=0.96128, valid AUC=0.94905, gap=0.01222
HistGradientBoosting | Fold 3: train AUC=0.96243, valid AUC=0.94506, gap=0.01737
HistGradientBoosting | Fold 4: train AUC=0.96218, valid AUC=0.94708, gap=0.01510
HistGradientBoosting | Fold 5: train AUC=0.96184, valid AUC=0.94830, gap=0.01354

OOF AUC: 0.94768
Mean valid AUC: 0.94773
Std valid AUC: 0.00171
Mean gap: 0.01409


In [10]:
all_baseline_results = pd.concat(
    [logreg_results, rf_results, hgb_results],
    ignore_index=True
)

model_summary = (
    all_baseline_results
    .groupby("model")
    .agg(
        mean_train_auc=("train_auc", "mean"),
        mean_valid_auc=("valid_auc", "mean"),
        std_valid_auc=("valid_auc", "std"),
        mean_gap=("gap", "mean"),
    )
    .sort_values("mean_valid_auc", ascending=False)
)

model_summary

,mean_train_auc,mean_valid_auc,std_valid_auc,mean_gap
model,,,,
HistGradientBoosting,0.961819,0.947730,0.001709,0.014089
Logistic Regression,0.913343,0.903673,0.002811,0.009669
Random Forest,0.889809,0.886494,0.002864,0.003315


### Baseline modeling insight

The baseline comparison allows us to evaluate both predictive performance and overfitting.  
A model with a very high training AUC but much lower validation AUC is likely too flexible.  
The preferred model should have strong validation AUC, low fold variance, and a reasonable train-validation gap.